In [1]:
import sys
sys.path.insert(0, '/mnt/disk_data/python_libs')

# **Импортируем необходимые модули**

In [2]:
import os
import time
import glob
import cv2

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# **Режим картинки на области**

In [ ]:
не запускаем

In [ ]:
class Slicer:
    def __init__(self, input_dir, output_dir):
        self.input_dir = input_dir
        self.output_dir = output_dir

    def process_dataset(self, tile_size):
        img_folder = os.path.join(self.input_dir, "images")
        mask_folder = os.path.join(self.input_dir, "masks")
        
        out_img_dir = os.path.join(self.output_dir, "images")
        out_mask_dir = os.path.join(self.output_dir, "masks")
        
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_mask_dir, exist_ok=True)

        # Получаем список всех изображений
        image_files = sorted(glob.glob(os.path.join(img_folder, "*")))
        mask_files = sorted(glob.glob(os.path.join(mask_folder, "*")))

        print(len(image_files), len(mask_files))

        for image_path, mask_path in zip(image_files, mask_files):
            print(f"Обработка пары: {image_path} и {mask_path}")
            self.slice_pair(image_path, mask_path, out_img_dir, out_mask_dir, tile_size)


    def slice_pair(self, img_path, mask_path, out_img_dir, out_mask_dir, tile_size):
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if not os.path.exists(mask_path):
            print(f"!!! ОШИБКА: Файл маски физически отсутствует по пути: {mask_path}")
            return

        height, width, _ = image.shape
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        y_cords = list(range(0, height, tile_size))
        if y_cords[-1] + tile_size > height:
            y_cords[-1] = height - tile_size

        x_cords = list(range(0, width, tile_size))
        if x_cords[-1] + tile_size > width:
            x_cords[-1] = width - tile_size

        # Нарезка
        count = 0
        for y in y_cords:
            for x in x_cords:
                img_crop = image[y:y + tile_size, x:x + tile_size]
                mask_crop = mask[y:y + tile_size, x:x + tile_size]

                tile_name = f"{base_name}_{count}.png"
                cv2.imwrite(os.path.join(out_img_dir, tile_name), img_crop)
                cv2.imwrite(os.path.join(out_mask_dir, tile_name), mask_crop)
                print(f"Данные сохранены в {os.path.join(out_img_dir, tile_name)} и {os.path.join(out_mask_dir, tile_name)}")
                count += 1

In [41]:
my_slicer = Slicer(input_dir='ResNetUNet-dataset/train/', output_dir='ResNetUNet-dataset_boxes/train/')
my_slicer.process_dataset(896)

my_slicer = Slicer(input_dir='ResNetUNet-dataset/test/', output_dir='ResNetUNet-dataset_boxes/test/')
my_slicer.process_dataset(896)

35 35
Обработка пары: ResNetUNet-dataset/train/images\2550374-2 10.JPG и ResNetUNet-dataset/train/masks\2550374-2 10.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_0.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_0.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_1.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_1.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_2.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_2.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_3.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_3.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_4.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_4.png
Данные сохранены в ResNetUNet-dataset_boxes/train/images\2550374-2 10_5.png и ResNetUNet-dataset_boxes/train/masks\2550374-2 10_5.png
Обработка пары: ResNetUNet-dataset/train/images\2550375-1 10.JPG и ResNetUN

# **Прдебработка**

In [3]:
class PreProcessingMethods:
    def __init__(self, image):
        # Конвертируем входной PIL в массив для OpenCV
        self.image_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

    # Подготовка изображеняи цветокорекция clahe
    def create_gray(self):
        image_gray = cv2.cvtColor(self.image_bgr, cv2.COLOR_BGR2GRAY)

        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        image_enhanced_gray = clahe.apply(image_gray)

        # Растягиваем на 3 канала (потому что ReNet принмаения 3 канальное изрбарежниян)
        image_rgb = cv2.cvtColor(image_enhanced_gray, cv2.COLOR_GRAY2RGB)

        # обратно возвращаем PIL Image
        return Image.fromarray(image_rgb)

    def create_hsv(self):
        image_hsv = cv2.cvtColor(self.image_bgr, cv2.COLOR_BGR2HSV)

        # обратно возвращаем PIL Image
        return Image.fromarray(image_hsv)

# **Содадим датасет**

In [4]:
class SegmentationDataset(Dataset):
  def __init__(self, root_dir, size_w: int, size_h: int, transform_img=None, transform_mask=None):
    self.root_dir = root_dir
    self.size_w = size_w
    self.size_h = size_h
    self.transform_img = transform_img
    self.transform_mask = transform_mask
    self.list_files()
    self.Preprocessor = PreProcessingMethods

  def list_files(self):
    images_dir = os.path.join(self.root_dir, 'images/')
    masks_dir = os.path.join(self.root_dir, 'masks/')

    images_dir_files = os.listdir(images_dir)
    mask_dir_files = os.listdir(masks_dir)

    image_names = sorted([img for img in images_dir_files if os.path.isfile(images_dir + img)])
    mask_names = sorted([mask for mask in mask_dir_files if os.path.isfile(masks_dir + mask)])
    
    self.images = [images_dir + i for i in image_names]
    self.masks = [masks_dir + i for i in mask_names]

  def __getitem__(self, idx):
    import numpy as np
    image = Image.open(self.images[idx]).convert("RGB")
    mask = Image.open(self.masks[idx]).convert("L")

    original_img = image.copy()

    image = image.resize((self.size_w, self.size_h))
    image = self.Preprocessor(image).create_gray()
    if self.transform_img: image = self.transform_img(image)
    
    if self.transform_mask: mask = self.transform_mask(mask)

    sample = {'image': image, 'mask': mask, "orig_img": np.array(original_img)}

    return sample

  def __len__(self):
    return len(self.images)


In [5]:
class ToBin:
  def __call__(self, image):
    res = image.clone()
    res[image > 0] = 1
    return res

In [6]:
base_model = resnet18(weights=ResNet18_Weights.DEFAULT)

In [7]:
base_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# **Модель**

In [8]:
class ResNetUNet(nn.Module):
  def __init__(self, freeze=True):
    super(ResNetUNet, self).__init__()

    base_model = resnet18(weights=ResNet18_Weights.DEFAULT)

    self.left_conv0 = nn.Sequential(base_model.conv1, base_model.bn1, base_model.relu) # 64 канала
    self.left_pool = base_model.maxpool
    self.left_conv1 = base_model.layer1 # 64 канала
    self.left_conv2 = base_model.layer2 # 128 каналов
    self.left_conv3 = base_model.layer3 # 256 каналов
    self.left_conv4 = base_model.layer4 # 512 каналов (bottleneck)

    if freeze:
      for param in [self.left_conv0, self.left_conv1, self.left_conv2, self.left_conv3, self.left_conv4]:
          for p in param.parameters():
              p.requires_grad = False

    self.right_conv1 = nn.Sequential(
      nn.Conv2d(768, 256, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(256, 128, (3, 3), padding=1),
      nn.ReLU(),
    )
    
    self.right_conv2 = nn.Sequential(
      nn.Conv2d(256, 128, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(128, 64, (3, 3), padding=1),
      nn.ReLU()
    )

    self.right_conv3 = nn.Sequential(
      nn.Conv2d(128, 64, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 32, (3, 3), padding=1),
      nn.ReLU()
    )

    self.right_conv4 = nn.Sequential(
      nn.Conv2d(96, 32, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(32, 1, (3, 3), padding=1),
      nn.Sigmoid()
    )

    self.up = nn.UpsamplingBilinear2d(scale_factor=2)
    self.pool = nn.MaxPool2d(kernel_size=2)

    
  def forward(self, x):
    left0 = self.left_conv0(x) # (H/2 x W/2 x 64)
    x = self.left_pool(left0) # (H/4 x W/4 x 64)
    left1 = self.left_conv1(x) # (H/4 x W/4 x 64)
    left2 = self.left_conv2(left1) # (H/8 x W/8 x 128)
    left3 = self.left_conv3(left2) # (H/16 x W/16 x 256)
    left4 = self.left_conv4(left3) # (H/32 x W/32 x 512)

    up1 = self.up(left4) # (H/16 x W/16 x 512)
    concat1 = torch.cat((up1, left3), 1) # (H/16 x W/16 x 768)
    right1 = self.right_conv1(concat1) # (H/16 x W/16 x 128)

    up2 = self.up(right1) # (H/8 x H/8 x 128)
    concat2 = torch.cat((up2, left2), 1) # (H/8 x H/8 x 256)
    right2 = self.right_conv2(concat2) # (H/8 x H/8 x 64)

    up3 = self.up(right2) # (H/4 x H/4 x 64)
    concat3 = torch.cat((up3, left1), 1) # (H/4 x H/4 x 128)
    right3 = self.right_conv3(concat3) # (H/4 x H/4 x 32)

    up4 = self.up(right3) # (H/2 x H/2 x 32)
    concat4 = torch.cat((up4, left0), 1) # (H/2 x H/2 x 96)
    right4 = self.right_conv4(concat4) # (H/2, W/2, 1)

    output = self.up(right4) # (H x W x 1)
    return output

# **Эпоха**

In [9]:
def make_epoch(model, criterion, dataloader, optimizer, device, alpha, metrics, epoch, num_epochs, fieldnames, loss_list):
    # Обучение
    since = time.time()
    print('Epoch {}/{}'.format(epoch, num_epochs))
    print('-' * 10)

    epoch_summary = {a: [0] for a in fieldnames}
    epoch_summary['epoch'] = epoch
    running_loss = {f: [] for f in fieldnames[1:]}

    model.train()
    
    for sample in dataloader['train']:
      inputs = sample['image'].to(device)
      masks = sample['mask'].to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, masks)
      loss.backward()
      optimizer.step()

      running_loss['Train_loss'].append(loss.item())      
      y_pred = (outputs.detach().cpu().numpy().ravel() > alpha).astype('uint8')
      y_true = masks.detach().cpu().numpy().ravel().astype('uint8')
      for name, metric in metrics.items():
        running_loss[f'Train_{name}'].append(metrics[name](y_true, y_pred))

    # Тестирование
    model.eval()
    with torch.no_grad():
      for sample in dataloader['test']:
        inputs = sample['image'].to(device)
        masks = sample['mask'].to(device)
        outputs = model(inputs)
        loss = criterion(outputs, masks)
        
        running_loss['Test_loss'].append(loss.item())
        y_pred = (outputs.detach().cpu().numpy().ravel() > alpha).astype('uint8')
        y_true = masks.detach().cpu().numpy().ravel().astype('uint8')
        for name, metric in metrics.items():
          running_loss[f'Test_{name}'].append(metric(y_true, y_pred))
    
    for field in fieldnames[1:]:
      epoch_summary[field] = np.mean(running_loss[field])
    loss_list.append(epoch_summary)

    print(epoch_summary)
    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

# **Функция обучения модели**

In [10]:
def train_model(model, criterion, dataloader, optimizer, alpha, metrics, num_epochs, loss_list):
  fieldnames = ['epoch', 'Train_loss', 'Test_loss'] + [f'Train_{m}' for m in metrics.keys()] + [f'Test_{m}' for m in metrics.keys()]

  # Кидаем на устройство
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  print(device)
  model.to(device)

  for epoch in range(len(loss_list) + 1, num_epochs + 1):
    make_epoch(model, criterion, dataloader, optimizer, device, alpha, metrics, epoch, num_epochs, fieldnames, loss_list)
  return model

# **Обучение**

не запускаем

In [44]:
'''
team006@team006:~/GeoAIprivate/dataset/train/images$ i=1; for f in *; do mv "$f" "img_$i.png"; ((i++)); done
team006@team006:~/GeoAIprivate/dataset/train/images$ cd ..
team006@team006:~/GeoAIprivate/dataset/train$ cd masks
team006@team006:~/GeoAIprivate/dataset/train/masks$ i=1; for f in *; do mv "$f" "img_$i.png"; ((i++)); done
team006@team006:~/GeoAIprivate/dataset/train/masks$
'''


#ROOT = '/home/team006/GeoAIprivate/'
ROOT = '/mnt/disk_data/GeoAIprivate/'
#ROOT = 'C:/Users/ASUS/source/repos/GeoAIprivate-main/'

BATCH_SIZE = 4
MAX_EPOCH = 35

size_w = 2272
size_h = 1704

size_w = 896
size_h = 896

size_w -= size_w%32
size_h -= size_h%32

transfrom_image = transforms.Compose([transforms.Resize((size_w, size_h)), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

transfrom_mask = transforms.Compose([transforms.Resize((size_w, size_h)), transforms.ToTensor(), ToBin()])

train_ds = SegmentationDataset(ROOT + "ResNetUNet-dataset_boxes/train/", size_w, size_h, transfrom_image, transfrom_mask)
#train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

test_ds = SegmentationDataset(ROOT + "ResNetUNet-dataset_boxes/test/", size_w, size_h, transfrom_image, transfrom_mask)
#test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

dataLoader = {"train": train_dl, "test": test_dl}
model = ResNetUNet()
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
alpha = 0.3
metrics = {'accuracy': accuracy_score}
loss_list = []
train_model(model, criterion, dataLoader, optimizer, alpha, metrics, MAX_EPOCH, loss_list)

cuda:0
Epoch 1/35
----------
{'epoch': 1, 'Train_loss': np.float64(0.594653586734016), 'Test_loss': np.float64(0.6470222933725878), 'Train_accuracy': np.float64(0.5108390164733345), 'Test_accuracy': np.float64(0.6578795763911033)}
Training complete in 0m 25s
Epoch 2/35
----------
{'epoch': 2, 'Train_loss': np.float64(0.48757136036764903), 'Test_loss': np.float64(0.5831634971228513), 'Train_accuracy': np.float64(0.7407242602736479), 'Test_accuracy': np.float64(0.6165422712053571)}
Training complete in 0m 25s
Epoch 3/35
----------
{'epoch': 3, 'Train_loss': np.float64(0.4520311625498646), 'Test_loss': np.float64(0.619176983833313), 'Train_accuracy': np.float64(0.7554814247245922), 'Test_accuracy': np.float64(0.6187797362375348)}
Training complete in 0m 25s
Epoch 4/35
----------
{'epoch': 4, 'Train_loss': np.float64(0.4266720582854073), 'Test_loss': np.float64(0.6715571826154535), 'Train_accuracy': np.float64(0.7727596913111131), 'Test_accuracy': np.float64(0.7012214448323719)}
Training c

ResNetUNet(
  (left_conv0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (left_pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (left_conv1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tr

# **Тест**

In [11]:
def denormalize(image):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image * 255, 0, 255).astype('uint8')
    return image

def generate_mask(model, dataloader, save_dir, device):
    model.eval()
    with torch.no_grad():
        for i, sample in enumerate(dataloader):
            inputs = sample['image'].to(device)
            outputs = model(inputs)
            originals = sample['orig_img']
            true_mask = sample['mask']
            
            for j in range(outputs.size(0)):
                mask = (outputs[j].squeeze().cpu().numpy() > alpha).astype('uint8') * 255
                Image.fromarray(mask).save(f"{save_dir}/output_{i * dataloader.batch_size + j}.png")
                Image.fromarray(denormalize(inputs[j].cpu().permute(1, 2, 0).numpy())).save(f"{save_dir}/img_{i * dataloader.batch_size + j}.png")
                Image.fromarray(originals[j].numpy().astype('uint8')).save(f"{save_dir}/orig_img_{i * dataloader.batch_size + j}.png")
                true_mask_np = true_mask[j].squeeze().cpu().numpy().astype('uint8')
                if true_mask_np.max() <= 1:
                    true_mask_np = true_mask_np * 255
                Image.fromarray(true_mask_np).save(f"{save_dir}/mask_{i * dataloader.batch_size + j}.png")

In [59]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
generate_mask(model, test_dl, "test_result", device)

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
generate_mask(model, train_dl, "train_result", device)

# **Сохранение**

In [12]:
def save_model(model, epoch, name):
    save_dir = ROOT + "weights/"
    file_path = os.path.join(save_dir, f"ResNetUNet{name}_[{epoch}].pth")

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
    }

    torch.save(checkpoint, file_path)

не запускаем

In [62]:
save_model(model, "v1", len(loss_list))

# **Загрузка**

In [13]:
def load_model(model, file_path):
  model = model(freeze=True)
  checkpoint = torch.load(file_path)
  model.load_state_dict(checkpoint['model_state_dict'])
  return model

In [14]:
ROOT = '/mnt/disk_data/GeoAIprivate/'
loaded_model = load_model(ResNetUNet, ROOT + "weights/" + "ResNetUNet35_[v1].pth")

/tmp/ipykernel_195941/1493272126.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path)


# **Применени**

In [15]:
from patchify import patchify, unpatchify

class GeoAI:
    def __init__(self, model_path: str):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model = load_model(ResNetUNet, model_path)
        self.model = self.model.half().to(self.device).eval()
        self.size_w = 896
        self.size_h = 896
        self.transform_img = transforms.Compose([transforms.Resize((self.size_w, self.size_h)), transforms.ToTensor(),
                                                 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        
    def predict(self, image_path, predict_path):
        print(self.device)
        Image.MAX_IMAGE_PIXELS = None
        image_bgr = cv2.imread(image_path)
        H, W, C = image_bgr.shape
        pad_h = (self.size_h - H % self.size_h) % self.size_h
        pad_w = (self.size_w - W % self.size_w) % self.size_w
        image_padded = np.pad(image_bgr, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')
        
        patches = patchify(image_padded, (self.size_w, self.size_h, 3), step=self.size_w)
        n_rows, n_cols = patches.shape[0], patches.shape[1]
        
        results = []
        self.model.eval()

        '''
        patches_flat = patches.reshape(-1, 256, 256, 3)
        
        with torch.no_grad():
            print(len(patches_flat))
            for i, patch in enumerate(patches_flat):
                gray_image = PreProcessingMethods(Image.fromarray(patch)).create_gray()
                tensor_patch = self.transform_img(gray_image).unsqueeze(0).to(self.device)
                
                output = self.model(tensor_patch)
                results.append(output.squeeze(0).detach())

                print(i)
        '''

        with torch.no_grad():
            for r in range(n_rows):
                for c in range(n_cols):
                    patch = patches[r, c, 0]
                    
                    gray_image = PreProcessingMethods(Image.fromarray(patch)).create_gray()
                    tensor_patch = self.transform_img(gray_image).unsqueeze(0).to(self.device).half()
                    
                    output = self.model(tensor_patch)
                    results.append(output.squeeze().detach())

                    torch.cuda.empty_cache()

        full_stack = torch.stack(results)
        stacked_results = full_stack.reshape(n_rows, n_cols, self.size_w, self.size_h).cpu().numpy()

        full_mask = unpatchify(stacked_results, (H + pad_h, W + pad_w))
        mask_to_save = (full_mask * 255).astype(np.uint8)
        cv2.imwrite(predict_path, mask_to_save)

In [16]:
model_path = ROOT + "weights/ResNetUNet35_[v1].pth"
image_path = ROOT + "Panorams/11.jpg"
predict_path = ROOT + "Panorams/11_output.png"

geo_ai = GeoAI(model_path=model_path)
pred = geo_ai.predict(image_path=image_path, predict_path=predict_path)


/tmp/ipykernel_195941/1493272126.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path)


cuda:0


In [21]:
import torch
import gc

# 1. Удаляем модель и списки
if 'geo_ai' in globals():
    del geo_ai
# Если вы создавали model отдельно:
# del model 

# 2. Принудительная сборка мусора Python
gc.collect()

# 3. Очистка кеша PyTorch (самое важное для GPU)
torch.cuda.empty_cache()

# 4. Проверка свободной памяти
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   | 460338 KiB |   3080 GiB |   3080 GiB |
|       from large pool |      0 B   | 452032 KiB |   3059 GiB |   3059 GiB |
|       from small pool |      0 B   |   9874 KiB |     20 GiB |     20 GiB |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   | 460338 KiB |   3080 GiB |   3080 GiB |
|       from large pool |      0 B   | 452032 KiB |   3059 GiB |